# Actividad 6 — Desigualdad espacial en la Región Metropolitana

**INF-497 · Análisis de Datos Espaciales**
**Sesión práctica · 29 abril (2 bloques)**

## Contexto

En la clase de hoy aplicamos a la RM las técnicas vistas en `06_desigualdad_espacial.ipynb` (basado en cap. 9 del libro *Geographic Data Science with Python*).

Trabajaremos con datos de la encuesta **CASEN 2017 y 2022** agregados a las **52 comunas** de la Región Metropolitana, ya preparados por el preámbulo. Tendremos dos definiciones de ingreso por comuna:

- **Ingreso autónomo per cápita** (`ypc_aut`): pre-transferencias estatales (sueldos, rentas, capital).
- **Ingreso total per cápita** (`ypc_tot`): post-transferencias (incluye subsidios y pensiones).

La gracia es que con esos datos podemos responder dos preguntas distintas:

1. *Temporal:* ¿la geografía de la desigualdad en RM cambió entre 2017 y 2022?
2. *Estructural:* ¿cuánto reducen las transferencias del Estado la desigualdad **espacial** (no solo la individual)?

## Reglas

- Suban este notebook con sus respuestas (código + texto en markdown) ejecutado.
- Las preguntas marcadas **Interpretación** se responden en celdas markdown — no basta con código.
- Si una comuna tiene `n < 200` en la muestra CASEN, el promedio comunal es ruidoso. Pueden discutir el efecto en sus respuestas.

## Requisito previo

El notebook **`06_actividad6_preambulo.ipynb`** generara el archivo `datos/external/casen_rm/casen_rm_comunas.gpkg`. Usan la CASEN 2017 Y 2022 de base. No lo necesitan ejecutar.

## 0. Setup

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

import esda
from libpysal import weights
from inequality.gini import Gini, Gini_Spatial
from inequality.theil import Theil, TheilD

sns.set_context("notebook")

gdf = gpd.read_file("../datos/external/casen_rm/casen_rm_comunas.gpkg")
print(f"{len(gdf)} comunas RM cargadas | CRS: {gdf.crs}")
gdf.head(3)

52 comunas RM cargadas | CRS: EPSG:32719


,cod_comuna,nom_comuna,nom_prov,region,ypc_aut_2017,ypc_aut_2022,ypc_tot_2017,ypc_tot_2022,mediana_aut_2017,mediana_aut_2022,n_2017,n_2022,geometry
0,13404,PAINE,MAIPO,13,285365.876995,320527.047313,353275.520186,446774.840194,200000.0,254583.0,420.0,456.0,"POLYGON ((350033.920 6265707.472, 350106.705 6..."
1,13402,BUIN,MAIPO,13,215822.686195,388319.809053,272555.328224,502456.613901,172000.0,280000.0,392.0,602.0,"POLYGON ((348666.339 6275861.274, 348652.409 6..."
2,13124,PUDAHUEL,SANTIAGO,13,235630.811008,400037.613058,297166.493830,515095.307407,195167.0,313750.0,935.0,1299.0,"POLYGON ((333540.425 6307203.281, 333624.972 6..."


---

## Ejercicio 1 — Exploración inicial

Antes de calcular métricas, vamos a ver con qué tipo de distribución estamos trabajando y qué patrones espaciales saltan a la vista.

### 1.A — Histogramas

Visualice en una grilla de 2x2  histogramas de las cuatro variables de ingreso (`ypc_aut_2017`, `ypc_aut_2022`, `ypc_tot_2017`, `ypc_tot_2022`). Agregue títulos claros.

In [2]:
# Tu código aquí


---

## Ejercicio 2 — Medidas globales de desigualdad

Calculen los tres índices clásicos para las cuatro combinaciones (autónomo / total × 2017 / 2022) y comparen.

### 2.A — Gini, Theil y Ratio 20:20

Construya un DataFrame `indices` con índice las 4 combinaciones (`aut_2017`, `aut_2022`, `tot_2017`, `tot_2022`) y columnas `gini`, `theil`, `ratio_20_20`.

Usen:
- `inequality.gini.Gini(valores).g` para Gini
- `inequality.theil.Theil(valores).T` para Theil
- Cálculo manual del ratio 20:20: `Q80 / Q20`

In [3]:
# Tu código aquí



### 2.B — Curvas de Lorenz comparadas

Grafiquen en **un solo eje** las 4 curvas de Lorenz (use distintos colores y tipos de línea — autónomo en sólido, total en punteado, 2017 en azul, 2022 en naranja, por ejemplo). Incluyan la diagonal de igualdad perfecta.

Pueden utilizar la función `lorenz(y)` 

```python
 def lorenz(y):
    y_sorted = np.sort(np.asarray(y))
    cum_y = (y_sorted / y_sorted.sum()).cumsum()
    cum_p = np.arange(1, len(y_sorted) + 1) / len(y_sorted)
    return cum_p, cum_y
```

In [4]:
# Tu código aquí


### 2.C — Interpretación

1. ¿La desigualdad **subió o bajó** entre 2017 y 2022 según los tres índices? ¿Coinciden?
2. Comparen ingreso autónomo vs total dentro del **mismo año**. ¿Cuánto reducen las transferencias del Estado la desigualdad **entre comunas**? Cuantifíquenlo (ej. caída porcentual del Gini).

**Respuesta 2.C:** *(escriban acá)*

---

## Ejercicio 3 — Descomposición regional del Theil por provincia

A diferencia del notebook 06 (que descomponía el Theil por las 8 regiones censales de EE.UU.), nosotros descompondremos por las **6 provincias** de la RM (Santiago, Cordillera, Maipo, Chacabuco, Talagante, Melipilla). La pregunta que respondemos: *¿cuánta de la desigualdad entre comunas RM viene de diferencias entre provincias, y cuánta de diferencias dentro de cada provincia?*

### 3.A — Calcular el Theil total y su descomposición

Para el ingreso autónomo per cápita 2022 (`ypc_aut_2022`):

1. Calculen el **Theil total** con `inequality.theil.Theil(...).T`.
2. Calculen `inequality.theil.TheilD(valores, gdf['nom_prov'].values)` y reporten:
   - `theil_entre = .bg`  — diferencias entre provincias
   - `theil_dentro = .wg` — diferencias dentro de provincias
   - `proporción entre = theil_entre / theil_total`

In [5]:
# Tu código aquí



### 3.B — Mapa de las provincias

Hagan un mapa de la RM coloreando cada comuna por su provincia:

```python
gdf.plot("nom_prov", categorical=True, legend=True, edgecolor="white", figsize=(9, 7))
```

Esto les da contexto visual: ¿cuántas comunas tiene cada provincia? ¿Hay provincias homogéneas y otras muy mezcladas?

In [6]:
# Tu código aquí


### 3.C — Interpretación

Respondan en una celda markdown:

1. ¿Qué porcentaje del Theil viene de diferencias **entre** provincias vs **dentro** de provincias? ¿Qué les dice ese número?

2. ¿Tiene sentido el resultado dado el mapa? 

3. ¿Qué pasaría con la proporción `entre / total` si en vez de provincias descompusiéramos por las 6 zonas geográficas tradicionales del Gran Santiago (oriente, surponiente, etc.)? **No lo calculen — solo razonen.**

**Respuesta 3.C:** *(escriban acá)*